# Notebook B: chia dữ liệu, mô hình và câu hỏi thứ ba

**Đề tài:** Dự báo trước một ngày điện tiêu thụ toàn quốc của Thổ Nhĩ Kỳ giữ được độ chính xác vào ngày lễ và ở các năm sau

**Sinh viên B** chạy sau khi Notebook A đã tạo file parquet, `sql/queries.sql` và `manifest.json`. Notebook này không cần kernel của Notebook A.

## Kiểm tra bàn giao

- **Làm gì:** đọc manifest và so mã băm của file parquet.
- **Vì sao:** bảo đảm hai notebook dùng đúng cùng một dữ liệu.
- **Kiểm tra:** ô in `handover ok`.

In [ ]:
# run once per machine, then comment out
# %pip install pandas numpy duckdb pyarrow scikit-learn lightgbm xgboost matplotlib scipy
import os, json, hashlib, numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
m = json.load(open('data/processed/manifest.json'))
assert hashlib.md5(open(m['file'], 'rb').read()).hexdigest() == m['md5'], 'parquet differs from the manifest, rerun Notebook A'
os.makedirs('report', exist_ok=True); print('handover ok', m['rows'], 'rows')

## Bước 4. Bảng đặc trưng và chia dữ liệu

- **Làm gì:** chạy lại `sql/queries.sql`, lấy bảng `feat`, chia tập huấn luyện và kiểm thử theo giao thức của đề tài.
- **Vì sao:** cách chia quyết định con số có đáng tin hay không; dòng `assert` là phép kiểm tra rò rỉ.
- **Kiểm tra:** ghi số dòng hai tập và số đặc trưng vào bảng mô tả thực nghiệm.

In [ ]:
# Step 4: forward split: 2021 to 2023 for training, 2024 for testing; 2025 is kept for RQ3
import duckdb
con = duckdb.connect(); con.execute(open('sql/queries.sql').read())
feat = con.execute("SELECT * FROM feat ORDER BY time").df(); feat['time'] = pd.to_datetime(feat.time); feat['target_time'] = pd.to_datetime(feat.target_time)
LOAD_F = ['consumption', 'cons_1h', 'cons_yesterday', 'same_hour_last_week', 'mean_24h', 'mean_7d', 'hour', 'dow', 'weekend', 'holiday', 'season_sin', 'season_cos']; WEATHER_F = ['temperature', 'temp_mean_24h', 'humidity', 'cloud', 'wind', 'rain']
X_cols = LOAD_F + WEATHER_F; TARGET = 'y'; NAIVE = 'same_hour_last_week'    # reference: the consumption of the same hour one week before the target
train = feat[feat.target_year <= 2023]; test = feat[feat.time.dt.year == 2024]; test = test[test.target_year == 2024]; test_2025 = feat[(feat.time.dt.year == 2025) & (feat.target_year == 2025)]
X_tr, y_tr, X_te, y_te = train[X_cols], train[TARGET], test[X_cols], test[TARGET]
assert train.target_time.max() < test.time.min()                       # leakage check: no training target lies inside the test year
MAPE = lambda y, p: float(100 * np.mean(np.abs(np.asarray(y) - np.asarray(p)) / np.asarray(y)))
print(len(feat), len(train), len(test), len(test_2025), len(X_cols), 'features', 'mean consumption train and test (MWh)', round(y_tr.mean()), round(y_te.mean()), 'MAPE of the weekly rule (%)', round(MAPE(y_te, test[NAIVE]), 2))

## Bước 4b. Hiểu tập huấn luyện và tập kiểm thử

Ba ô dưới đây kiểm tra hai tập có giống nhau không và đặc trưng nào mạnh nhất, trước khi huấn luyện bất kỳ mô hình nào.

### 4b.1 Mục tiêu ở hai tập

- **Làm gì:** so phân phối của mục tiêu giữa tập huấn luyện và tập kiểm thử.
- **Vì sao:** hai phân phối lệch nhau giải thích vì sao mô hình cây ngoại suy kém.

> **Nhận xét của nhóm (bắt buộc, một câu có số):** ...

In [ ]:
import matplotlib.pyplot as plt
def bar_chart(labels, values, ylabel, path, highlight=None, fmt='{:.1f}'):
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    colors = ['#F4A261' if i == highlight else '#1B6B6D' for i in range(len(values))]
    bars = ax.bar([str(l) for l in labels], values, color=colors)
    ax.bar_label(bars, labels=[fmt.format(v) for v in values], fontweight='bold', padding=2, fontsize=8 if len(values) > 7 else 10)
    ax.set_ylabel(ylabel); ax.set_ylim(0, max(values) * 1.2)
    ax.spines[['top', 'right']].set_visible(False)          # no top and right spines
    if len(values) > 7: ax.tick_params(axis='x', labelsize=7, rotation=30); plt.setp(ax.get_xticklabels(), ha='right')
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.show()

import matplotlib.pyplot as plt
def nice(v, label):                                     # readable numbers on the bars
    v = np.asarray(v, dtype=float); m = np.nanmax(np.abs(v))
    return (v / 1e6, label + ' (million)', '{:.2f}') if m >= 1e6 else (v / 1e3, label + ' (thousand)', '{:.1f}') if m >= 1e4 else (v, label, '{:.2f}' if m < 100 else '{:.0f}')
ytr, yte = pd.Series(np.asarray(y_tr, dtype=float)), pd.Series(np.asarray(y_te, dtype=float)); BINARY = ytr.nunique() <= 2
print('train rows', len(ytr), 'test rows', len(yte), 'features', X_tr.shape[1])
if BINARY: print('positive share: train %.3f, test %.3f' % (ytr.mean(), yte.mean()))
else:
    fig, ax = plt.subplots(figsize=(6.5, 3)); lo, hi = min(ytr.quantile(.01), yte.quantile(.01)), max(ytr.quantile(.99), yte.quantile(.99)); bins = np.linspace(lo, hi, 50)
    ax.hist(ytr.clip(lo, hi), bins=bins, density=True, alpha=.6, color='#1B6B6D', label='Train'); ax.hist(yte.clip(lo, hi), bins=bins, density=True, alpha=.6, color='#F4A261', label='Test')
    ax.legend(frameon=False); ax.set_xlabel('Target'); ax.set_ylabel('Density'); ax.spines[['top', 'right']].set_visible(False); fig.tight_layout(); fig.savefig('report/fig_split_target.png', dpi=200); plt.show()
    print(pd.DataFrame({'train': ytr.describe(), 'test': yte.describe()}).round(3))

### 4b.2 Đặc trưng lệch giữa hai tập

- **Làm gì:** độ lệch trung bình chuẩn hoá của từng đặc trưng giữa hai tập.
- **Kiểm tra:** đặc trưng lệch trên 0,5 độ lệch chuẩn là nguồn rủi ro, nêu trong mục Limitations.

> **Nhận xét của nhóm (bắt buộc, một câu có số):** ...

In [ ]:
A_, B_ = X_tr.astype(float), X_te.astype(float); sd = A_.std().replace(0, np.nan)
shift = ((B_.mean() - A_.mean()) / sd).abs().sort_values(ascending=False).rename('abs_standardised_mean_difference')
shift.to_csv('report/table_split_shift.csv'); display(shift.head(10).round(3).to_frame())
print('features whose test mean lies more than 0.5 training standard deviations away:', int((shift > 0.5).sum()), 'of', len(shift))

### 4b.3 Sức mạnh của từng đặc trưng

- **Làm gì:** tương quan hạng giữa từng đặc trưng và mục tiêu trên tập huấn luyện.
- **Vì sao:** nếu một đặc trưng đã gần đủ, mô hình phức tạp sẽ không hơn nhiều, và đó là thông tin để chọn câu hỏi.

> **Nhận xét của nhóm (bắt buộc, một câu có số):** ...

In [ ]:
sig = A_.sample(min(len(A_), 20_000), random_state=0); ys = pd.Series(np.asarray(y_tr, dtype=float), index=X_tr.index).loc[sig.index]
strength = sig.corrwith(ys, method='spearman').abs().sort_values(ascending=False).rename('abs_rank_correlation_with_target')
strength.to_csv('report/table_feature_strength.csv'); display(strength.head(10).round(3).to_frame())
print('strongest single feature:', strength.index[0], round(float(strength.iloc[0]), 3), '-> a model must beat what this feature already gives')

## Bước 6a. So sánh mô hình (RQ2)

- **Làm gì:** huấn luyện mười mô hình với năm seed trên cùng cách chia.
- **Vì sao:** so nhiều họ mô hình để kết luận không phụ thuộc một kiểu mô hình; năm seed cho độ lệch chuẩn.
- **Kiểm tra:** chênh lệch nhỏ hơn độ lệch chuẩn thì chưa được kết luận mô hình nào hơn. Bảng lưu ở `report/table_rq2.csv`.

In [ ]:
# Step 6a: one reference and ten regressors from four families, five seeds, same forward split by year
import numpy as np, pandas as pd, time
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error as MAE, mean_squared_error as MSE, r2_score
sc = lambda m: make_pipeline(StandardScaler(), m)       # scaling for linear, distance and neural models
models = {
    'Ridge':          lambda s: sc(Ridge(alpha=1.0)),
    'Elastic Net':    lambda s: sc(ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000, random_state=s)),
    'k-NN':           lambda s: sc(KNeighborsRegressor(n_neighbors=15)),
    'Decision Tree':  lambda s: DecisionTreeRegressor(max_depth=12, min_samples_leaf=10, random_state=s),
    'Random Forest':  lambda s: RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=s),
    'Extra Trees':    lambda s: ExtraTreesRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=s),
    'HistGB':         lambda s: HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=s),
    'LightGBM':       lambda s: LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8,
                                              subsample_freq=1, colsample_bytree=0.8, random_state=s, verbose=-1),
    'XGBoost':        lambda s: XGBRegressor(n_estimators=500, learning_rate=0.03, max_depth=6, subsample=0.8,
                                             colsample_bytree=0.8, tree_method='hist', random_state=s),
    'Small MLP':      lambda s: sc(MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, early_stopping=True,
                                                random_state=s)),
}
naive = test[NAIVE]                                     # reference: the consumption of the same hour one week before the target
rows, fitted = [['Same hour last week', MAE(y_te, naive), 0, MSE(y_te, naive) ** 0.5, r2_score(y_te, naive), 0]], {}
for name, make in models.items():
    maes, rmses, r2s, secs = [], [], [], []
    for seed in range(5):
        m = make(seed); t0 = time.perf_counter(); m.fit(X_tr, y_tr); secs.append(time.perf_counter() - t0)
        p = m.predict(X_te); maes.append(MAE(y_te, p)); rmses.append(MSE(y_te, p) ** 0.5); r2s.append(r2_score(y_te, p))
    fitted[name] = m                                    # keep the last fitted model of each family for Step 6b
    rows.append([name, np.mean(maes), np.std(maes), np.mean(rmses), np.mean(r2s), np.mean(secs)])
res = pd.DataFrame(rows, columns=['model', 'MAE', 'sd', 'RMSE', 'R2', 'train_s'])
res.to_csv('report/table_rq2.csv', index=False); print(res.round(4))

### Hình RQ2

- **Làm gì:** vẽ kết quả các mô hình, tô màu nhấn mô hình tốt nhất.
- **Kiểm tra:** số trên cột khớp với `report/table_rq2.csv`.

In [ ]:
import matplotlib.pyplot as plt
def bar_chart(labels, values, ylabel, path, highlight=None, fmt='{:.1f}'):
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    colors = ['#F4A261' if i == highlight else '#1B6B6D' for i in range(len(values))]
    bars = ax.bar([str(l) for l in labels], values, color=colors)
    ax.bar_label(bars, labels=[fmt.format(v) for v in values], fontweight='bold', padding=2, fontsize=8 if len(values) > 7 else 10)
    ax.set_ylabel(ylabel); ax.set_ylim(0, max(values) * 1.2)
    ax.spines[['top', 'right']].set_visible(False)          # no top and right spines
    if len(values) > 7: ax.tick_params(axis='x', labelsize=7, rotation=30); plt.setp(ax.get_xticklabels(), ha='right')
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.show()

bar_chart(res.model, res.iloc[:, 1], res.columns[1], 'report/fig_rq2_models.png', highlight=int(res.iloc[:, 1].idxmin()), fmt='{:.2f}')

## Cổng khả thi: có nên xoay câu hỏi không

- **Làm gì:** so mô hình tốt nhất với mốc đơn giản và in ra một trong ba kết luận.
- **Vì sao:** nếu mô hình không hơn mốc hoặc bài toán quá dễ, nhóm xoay câu hỏi ngay ở đây thay vì cố chỉnh mô hình. Xoay câu hỏi nghĩa là đổi điều được hỏi cho khớp với điều dữ liệu trả lời được, không phải chọn lại số liệu cho đẹp.

**Các hướng xoay của đề tài này:**

1. Mô hình hơn quy tắc tuần trước rõ rệt và sai số ngày lễ giảm mạnh là kết quả dương: giữ trục ngày lễ.
2. Nếu phản biện đòi so với bài học sâu: nêu MAPE của bài mốc trên cùng nguồn dữ liệu và nhấn mạnh mô hình bảng chạy trong vài giây.
3. Nếu muốn thêm một kết quả: đổi mục tiêu sang giá điện và cảnh báo giờ giá tăng vọt, cột is_price_spike đã có trong file.

**Quyết định của nhóm (giữ câu hỏi hay xoay theo hướng nào, vì sao):** ...

In [ ]:
ref, best = res.iloc[0], res.iloc[1:].sort_values('MAE').iloc[0]; skill = 100 * (1 - best.MAE / ref.MAE)
print('reference %s: MAE %.4f | best model %s: MAE %.4f | improvement %.1f%%' % (ref.model, ref.MAE, best.model, best.MAE, skill))
verdict = 'GOOD: the model clearly beats the reference, keep RQ2 as planned' if skill >= 10 else 'SMALL: report the reference as a strong baseline and move the contribution to RQ3' if skill >= 2 else 'NONE: the reference wins, pivot the question (see the options above) and report this result honestly'
print(verdict); pd.Series({'reference_MAE': ref.MAE, 'best_model': best.model, 'best_MAE': best.MAE, 'improvement_pct': skill, 'verdict': verdict}).to_csv('report/table_feasibility_gate.csv')

## Bước 6b. Câu hỏi thứ ba (RQ3)

- **Làm gì:** Dùng mười mô hình đã huấn luyện để chấm riêng ngày lễ và năm 2025, rồi huấn luyện lại bốn mô hình đại diện khi thêm dữ liệu 2024 và với ba mức thông tin thời tiết.
- **Vì sao:** đây là đóng góp chính của bài; mỗi bảng `report/table_rq3_*.csv` ứng với một ý trong mục Results.
- **Kiểm tra:** đọc từng bảng in ra và viết một câu kết luận có số cho mỗi bảng. Nếu kết quả trái dự kiến trong đề cương thì báo đúng như vậy và tìm nguyên nhân, không chỉnh code cho ra kết quả mong muốn.

In [ ]:
# Step 6b: RQ3, how large is the error on public holidays, does the model of 2023 still serve 2025, does retraining help, and what is the weather worth?
FAM = ['Ridge', 'Random Forest', 'LightGBM', 'Small MLP']; rows = []
for name, m in [('Same hour last week', None)] + list(fitted.items()):     # (1) ordinary days against public holidays, in % of the true consumption
    p = test[NAIVE].to_numpy() if m is None else m.predict(X_te); h = (test.holiday == 1).to_numpy(); rows.append([name, MAPE(y_te, p), MAPE(y_te[~h], p[~h]), MAPE(y_te[h], p[h])])
hd = pd.DataFrame(rows, columns=['model', 'MAPE_all', 'MAPE_ordinary_days', 'MAPE_holidays']); hd.to_csv('report/table_rq3_holidays.csv', index=False); print(hd.round(2)); print('holiday hours in the test year:', int((test.holiday == 1).sum()))
rows = [['Same hour last week', MAPE(test_2025[TARGET], test_2025[NAIVE]), np.nan]]; tr24 = feat[feat.target_year <= 2024]      # (2) one more year ahead: model frozen at the end of 2023 against a model retrained with 2024
for name in FAM: rows.append([name, MAPE(test_2025[TARGET], fitted[name].predict(test_2025[X_cols])), MAPE(test_2025[TARGET], models[name](0).fit(tr24[X_cols], tr24[TARGET]).predict(test_2025[X_cols]))])
ag = pd.DataFrame(rows, columns=['model', 'MAPE_2025_frozen_2023', 'MAPE_2025_retrained_with_2024']); ag.to_csv('report/table_rq3_2025.csv', index=False); print(ag.round(2))
rows = []                                                              # (3) no weather, weather measured now, and the temperature of the target hour (a perfect forecast, the upper bound)
for name in FAM:
    row = [name]
    for cols in [LOAD_F, X_cols, X_cols + ['temp_at_target']]: row.append(MAPE(y_te, models[name](0).fit(train[cols], y_tr).predict(test[cols])))
    rows.append(row)
wt = pd.DataFrame(rows, columns=['model', 'MAPE_no_weather', 'MAPE_weather_now', 'MAPE_with_temperature_at_target']); wt.to_csv('report/table_rq3_weather.csv', index=False); print(wt.round(2))

## Bước 6c. Kiểm tra bổ sung và bảng tham số

- **Làm gì:** chạy phép kiểm tra bổ sung của đề tài và lưu tham số mô hình.
- **Vì sao:** bảng này đi vào phần thảo luận và phụ lục của bài.
- **Kiểm tra:** `report/params_best.json` tồn tại.

In [ ]:
# Step 6c: the daily peak hour, error by month, strongest features, and parameters
best = res.iloc[1:].sort_values('MAE').model.iloc[0]; T = test.assign(p=fitted[best].predict(X_te), day=test.target_time.dt.date)
pk = T.groupby('day').apply(lambda g: pd.Series({'true_peak': g[TARGET].max(), 'model_peak': g.p.max(), 'rule_peak': g[NAIVE].max(), 'hour_true': g.loc[g[TARGET].idxmax(), 'hour'], 'hour_model': g.loc[g.p.idxmax(), 'hour']}))
peak = pd.DataFrame([['Same hour last week', MAPE(pk.true_peak, pk.rule_peak), np.nan], [best, MAPE(pk.true_peak, pk.model_peak), float(100 * ((pk.hour_true - pk.hour_model).abs() <= 1).mean())]], columns=['model', 'MAPE_of_daily_peak', 'peak_hour_within_one_hour_pct'])
peak.to_csv('report/table_rq3_daily_peak.csv', index=False); print(best); print(peak.round(2))
mo = T.groupby(T.target_time.dt.month).apply(lambda g: pd.Series({'hours': len(g), 'mean_mwh': g[TARGET].mean(), 'MAPE_last_week': MAPE(g[TARGET], g[NAIVE]), 'MAPE_model': MAPE(g[TARGET], g.p)})); mo.to_csv('report/table_rq3_by_month.csv'); print(mo.round(2))
from sklearn.inspection import permutation_importance
s = test.sample(min(4000, len(test)), random_state=0); imp = permutation_importance(fitted[best], s[X_cols], s[TARGET], scoring='neg_mean_absolute_error', n_repeats=3, random_state=0)
top = pd.DataFrame({'feature': X_cols, 'rise_in_MAE': imp.importances_mean}).sort_values('rise_in_MAE', ascending=False).head(10); top.to_csv('report/table_rq3_feature_importance.csv', index=False); print(top.round(1))
import json
json.dump({'best': best, 'target': 'national consumption 24 hours ahead', 'params': {k: str(v) for k, v in models[best](0).get_params().items()}}, open('report/params_best.json', 'w'), indent=2)

### Hình RQ3

- **Làm gì:** vẽ hai hình RQ3 từ các bảng `report/table_rq3_*.csv` bằng hàm `bar_chart` ở trên, theo mẫu Hình 4 và Hình 5 của đề cương.
- **Kiểm tra:** mỗi hình một ý, chữ tiếng Anh, số in đậm trên cột.

In [ ]:
import glob
for f in sorted(glob.glob('report/table_rq3_*.csv')):
    t = pd.read_csv(f); numc = [c for c in t.columns[1:] if pd.api.types.is_numeric_dtype(t[c])]
    print(f, t.shape)
    if 2 <= len(t) <= 14 and numc and t[numc[0]].notna().all() and t[numc[0]].min() >= 0 and t[numc[0]].max() > 0:
        v, lab, fm = nice(t[numc[0]], numc[0]); bar_chart(t.iloc[:, 0].astype(str), v, lab, f.replace('table_', 'fig_').replace('.csv', '.png'), highlight=int(v.argmax()), fmt=fm)

## Test case và chẩn đoán

- **Làm gì:** kiểm tra tự động các sản phẩm của notebook.
- **Kiểm tra:** mọi `assert` qua. Sinh viên A chạy lại notebook này và ghi vào ô Đối chứng.

In [ ]:
assert os.path.exists('report/table_rq2.csv') and os.path.exists('report/params_best.json')
assert len(glob.glob('report/table_rq3_*.csv')) >= 2, 'RQ3 tables missing'
if not res.iloc[1:, 1].min() <= res.iloc[0, 1]: print('WARNING: no model beats the simple reference, check features and split, then report it')
print(sorted(os.path.basename(f) for f in glob.glob('report/*')))
print('Notebook B: all checks passed')

### Đối chứng

| Người chạy lại | Ngày | Metric chính khớp | Ghi chú |
|---|---|---|---|
| | | | |

### Chẩn đoán

Nếu mô hình tốt nhất không hơn mốc đơn giản hoặc kết quả RQ3 trái dự kiến, ghi nguyên nhân có bằng chứng (thiếu đặc trưng, dữ liệu ít, lệch giữa hai tập) vào `report/table_diagnosis.csv`.

### Báo cáo tiến độ (dán vào kênh nhóm trước buổi báo cáo)

1. Việc đã xong, kèm tên file và số dòng, bảng, hình:
2. Con số chính mới có:
3. Vướng mắc và điều đã thử:
4. Việc sẽ làm đến lần báo cáo sau, ai làm:
5. Prompt AI đáng chú ý và Human Delta: